# Linear Regression Objective 2.1: 

### Task

Building linear regression model that evaluate prediciton the goal difference between two opposing teams across 104 FIFA World Cup Matches using eigh pre-match explanator variables.

### Outcome

Goal Difference = Goals scored by Team A - Goals scored by Team B

The dataset sourced from the FIFA official statistics platform, The Stats Don't Lie, and FBref. all the explanatory variables are created only for information available before the focal match and rolling statistics use earlier tournament matches only to prevent target leakage.

## This code file contains:

data validation, EDA, inferential testing, correlation, baseline OLS, systematic model experimentation, ridge comparison, residual diagnostics, influece analysis, senstitive checks, visualisation, and final recommendation.

In [19]:
# importing the library

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [22]:
df = pd.read_csv('Objective_2_1_Dataset.csv')
df.head(10)

,match_id,stage,team_A,team_B,goals_A,goals_B,goal_difference,X1_ppg_diff,X2_avg_goals_for_diff,X3_avg_goals_against_diff,X4_win_rate_diff,X5_clean_sheet_rate_diff,X6_goal_diff_per_match_diff,X7_prior_matches_diff,X8_host_advantage_diff
0,1,Group A,Mexico,South Africa,2,0,2,0.0,0.0,0.0,0.0,0.0,0.0,0,1
1,2,Group A,South Korea,Czechia,2,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,3,Group B,Canada,Bosnia and Herzegovina,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,4,Group D,United States,Paraguay,4,1,3,0.0,0.0,0.0,0.0,0.0,0.0,0,1
4,5,Group B,Qatar,Switzerland,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
5,6,Group C,Brazil,Morocco,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
6,7,Group C,Haiti,Scotland,0,1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,0
7,8,Group D,Australia,Türkiye,2,0,2,0.0,0.0,0.0,0.0,0.0,0.0,0,0
8,9,Group E,Germany,Curaçao,7,1,6,0.0,0.0,0.0,0.0,0.0,0.0,0,0
9,10,Group F,Netherlands,Japan,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0


In [23]:
# reading the data type
df.dtypes

match_id                         int64
stage                              str
team_A                             str
team_B                             str
goals_A                          int64
goals_B                          int64
goal_difference                  int64
X1_ppg_diff                    float64
X2_avg_goals_for_diff          float64
X3_avg_goals_against_diff      float64
X4_win_rate_diff               float64
X5_clean_sheet_rate_diff       float64
X6_goal_diff_per_match_diff    float64
X7_prior_matches_diff            int64
X8_host_advantage_diff           int64
dtype: object

### Dataset Specification:
- 104 rows, one per actual World Cup 2026 match;
- exactly 8 explanatory variables;
- target = goal_difference;
- explanatory variables are based only on information available before the match.

In [25]:
predictors = [
    "X1_ppg_diff",
    "X2_avg_goals_for_diff",
    "X3_avg_goals_against_diff",
    "X4_win_rate_diff",
    "X5_clean_sheet_rate_diff",
    "X6_goal_diff_per_match_diff",
    "X7_previous_goal_margin_diff",
    "X8_host_advantage_diff",
]
target = "goal_difference"

assert len(df) == 104
assert len(predictors) == 8
assert df[predictors + [target]].isna().sum().sum() == 0
assert (df["goal_difference"] == df["goals_A"] - df["goals_B"]).all()

quality_check = pd.DataFrame({
    "Check": [
        "Match rows",
        "Explanatory variables",
        "Missing predictor/target values",
        "Target calculation errors",
        "Constant predictors"
    ],
    "Required": [104, 8, 0, 0, 0],
    "Observed": [
        len(df),
        len(predictors),
        int(df[predictors + [target]].isna().sum().sum()),
        int((df["goal_difference"] != df["goals_A"] - df["goals_B"]).sum()),
        int(sum(df[c].nunique() <= 1 for c in predictors))
    ]
})
quality_check["Status"] = np.where(
    quality_check["Required"] == quality_check["Observed"], "PASS", "CHECK"
)
quality_check

KeyError: "['X7_previous_goal_margin_diff'] not in index"